# Ablations and native export parity: state of the art

**Level:** 3  
**Prerequisites:** Ablations and native export parity level 1  
**Reading time:** 25 minutes

## Objectives

The learning objective is to explain the problem and its data, then inspect and, at higher levels, execute the released method on its matched task. A complete evaluation must test whether each proposed component contributes and whether optimized model exports preserve the reference scorer. A matching score is numerical evidence only; it does not establish safer decisions, better task outcomes or generalization.

## Setup

The versions below are pinned in `notebooks/requirements.txt` and are also recorded in this notebook's metadata.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Data

The notebook reads committed product artifacts. Owner: Neuraxis. Repository: https://github.com/fsantibanezleal/CAOS_Neuraxis. License: Apache-2.0. The JSON is a first-party, frozen experiment artifact; no external benchmark data are redistributed in this notebook.

This notebook reads `data/derived/index.json` and the listed trace artifacts. The method run uses an isolated Reflexmesh case workspace. For learned methods, fetch the exact public release assets with `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000`, then set `NEURAXIS_CHECKPOINTS` to that folder. The release manifest verifies asset hashes. Planner methods M09/M10 are read from frozen traces unless `NEURAXIS_NOTEBOOK_MODEL` explicitly names a configured local Ollama model.

Artifacts for this lesson:

- `data/derived/index.json`
- `data/derived/research.json`
- `docs/evaluation/corrected-results.json`
- `data/derived/episodes/M03-C07-nominal-40000.json`
- `data/derived/episodes/M04-C07-nominal-40000.json`
- `data/derived/episodes/M11-C07-nominal-40000.json`
- `data/derived/episodes/M12-C07-nominal-40000.json`

## Method

Audit the canonical index digest against the research report, compare all four matched M12 ablations with the full controller, then inspect the native M03/M04 export vectors, threshold coverage, error tolerance and warmed latency scope.

For matched outcome \(Y\), the ablation contrast is \(\Delta=Y_{full}-Y_{ablated}\); paired seeds and fixed task templates determine the unit of uncertainty. Export parity is \(\max_i |p_i^{native}-p_i^{reference}|\le\epsilon\), a separate numerical criterion.

### Check the pinned analysis environment.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verify the audit and each trace's bytes before interpreting outcomes.

In [2]:
# Read committed evidence and use the released API; never define another policy.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M03', 'M04', 'M11', 'M12']
sample_case = 'C07'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C07 | seed: 40000
Methods: M03, M04, M11, M12


### Inspect the original decisions and outcomes

In [3]:
# Summarize the recorded first action, authorized candidates, and independent oracle outcome.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M03 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M04 | action= a-cancel | reason= calibrated_demonstration_preference | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M11 | action= a-cancel | reason= learned_incremental_benefit_gate | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0
M12 | action= a-cancel | reason= urgent_cancellation_during_deliberation | allowed= ['a-ack_event', 'a-cancel', 'a-refresh', 'a-report', 'a-wait', 'a-wait_dependency']
  success= 1 | steps= 1 | violations= 0


### Execute the released policies and compare the run

In [4]:
# Planner outcomes come from frozen traces; no LLM call is implicit.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M03 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M04 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M11 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}
M12 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-cancel'}


### Compare the full sequence, not only the first action

In [5]:
# Sequence equality is diagnostic; the independent verifier determines task success.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M03 a-cancel | terminal= succeeded | verified= 1
M04 a-cancel | terminal= succeeded | verified= 1
M11 a-cancel | terminal= succeeded | verified= 1
M12 a-cancel | terminal= succeeded | verified= 1


### Recompute per-variant outcomes from every trace in the case

In [6]:
# Read every seed for the selected case and keep each failed episode in the denominator.
from collections import defaultdict
groups = defaultdict(list)
for row in index['episodes']:
    if row['case_id'] not in (sample_case,) or row['method_id'] not in selected_methods:
        continue
    raw = (ROOT / 'data/derived' / row['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == row['sha256']
    episode = json.loads(raw)
    groups[(row['method_id'], row['variant'])].append(int(bool(episode['metrics']['task_success'])))
for (method_id, variant), values in sorted(groups.items()):
    print(method_id, variant, 'successes=', sum(values), 'episodes=', len(values), 'rate=', round(sum(values)/len(values), 3))

M03 boundary successes= 10 episodes= 10 rate= 1.0
M03 delayed_observation successes= 10 episodes= 10 rate= 1.0
M03 duplicate_event successes= 10 episodes= 10 rate= 1.0
M03 nominal successes= 10 episodes= 10 rate= 1.0
M03 stale_conflict successes= 10 episodes= 10 rate= 1.0
M03 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M04 boundary successes= 10 episodes= 10 rate= 1.0
M04 delayed_observation successes= 10 episodes= 10 rate= 1.0
M04 duplicate_event successes= 10 episodes= 10 rate= 1.0
M04 nominal successes= 10 episodes= 10 rate= 1.0
M04 stale_conflict successes= 10 episodes= 10 rate= 1.0
M04 unavailable_dependency successes= 10 episodes= 10 rate= 1.0
M11 boundary successes= 10 episodes= 10 rate= 1.0
M11 delayed_observation successes= 10 episodes= 10 rate= 1.0
M11 duplicate_event successes= 10 episodes= 10 rate= 1.0
M11 nominal successes= 10 episodes= 10 rate= 1.0
M11 stale_conflict successes= 10 episodes= 10 rate= 1.0
M11 unavailable_dependency successes= 10 episodes= 10

### Plot the variant stress without pooling families

In [7]:
# Each point summarizes the seeds from one case variant, not the full benchmark.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
variants = sorted(set(key[1] for key in groups))
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for method_id in selected_methods:
    rates = [sum(groups[(method_id, variant)])/len(groups[(method_id, variant)]) if groups[(method_id, variant)] else float('nan') for variant in variants]
    ax.plot(variants, rates, marker='o', label=method_id)
ax.set_ylim(-0.05, 1.05)
ax.set_ylabel('Verified success rate')
ax.set_title(sample_case + ' by variant')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=.25)
ax.legend()
display(fig)
plt.close(fig)

<Figure size 900x450 with 1 Axes>

### Recompute the matched ablations

In [8]:
# Research is bound to the canonical index digest and preserves cell-level pairing.
research = json.loads((ROOT / 'data/derived/research.json').read_text(encoding='utf-8'))
assert research['core']['index_sha256'] == hashlib.sha256(index_bytes).hexdigest()
full = next(row for row in research['core']['methods'] if row['method_id']=='M12')
print('Full M12:', full['successes'], '/', full['episodes'], '| rate=', full['success_rate'])
for name, result in research['ablations'].items():
    ablated = next(row for row in result['methods'] if row['method_id']=='M12')
    paired = result['paired_full_minus_ablation']
    assert paired['paired_cells'] == ablated['episodes'] == 1200
    assert ablated['successes'] == full['successes']
    print(name, '| successes=', ablated['successes'], '/', ablated['episodes'], '| paired delta=', paired['success_difference_left_minus_right'], '| seed bootstrap95=', paired['paired_seed_bootstrap95'], '| full-minus-ablation wall ms=', round(paired['mean_wall_difference_ms_left_minus_right'],3), '| scope=', paired['scope'])
assert len(research['ablations']) == 4

Full M12: 1160 / 1200 | rate= 0.9666666666666667
no_memory | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 2.588 | scope= designed shared templates; small seed-group population; compute budgets are not equalized
no_effects | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 39.28 | scope= designed shared templates; small seed-group population; compute budgets are not equalized
no_deferral | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 131.89 | scope= designed shared templates; small seed-group population; compute budgets are not equalized
no_invalidation | successes= 1160 / 1200 | paired delta= 0.0 | seed bootstrap95= [0.0, 0.0] | full-minus-ablation wall ms= 5.0 | scope= designed shared templates; small seed-group population; compute budgets are not equalized


### Audit M03/M04 numerical parity

In [9]:
# Numerical parity does not establish safety, decision calibration, or generalization.
for method_id, result in research['native_parity']['results'].items():
    assert result['passed'] and result['vectors'] == 484 and result['split_thresholds'] == 76
    assert result['max_abs_probability_error'] <= result['tolerance']
    timing = result['warmed_latency_ms']
    print(method_id, '| vectors=',result['vectors'],'| thresholds=',result['split_thresholds'],'| max error=',result['max_abs_probability_error'],'<=',result['tolerance'])
    print('  warmed p50/p95 ms native=',timing['native_cached']['50'],timing['native_cached']['95'],'| Python=',timing['python_reference']['50'],timing['python_reference']['95'])
    print('  scope:',result['warmed_timing_scope'])
print(research['native_parity']['note'])

M03 | vectors= 484 | thresholds= 76 | max error= 3.1308289294429414e-14 <= 1e-05
  warmed p50/p95 ms native= 0.009 0.0118 | Python= 0.10905000000000001 0.1334849999999999
  scope: Native: packed feature serialization + cached Rust scoring + return list; Python: fitted model predict + Platt; both exclude shared feature construction and effect execution
M04 | vectors= 484 | thresholds= 76 | max error= 3.7825163745619506e-07 <= 1e-05
  warmed p50/p95 ms native= 0.0185 0.0231 | Python= 0.2917 0.38110499999999975
  scope: Native: packed feature serialization + cached Rust scoring + return list; Python: fitted model predict + Platt; both exclude shared feature construction and effect execution
Numerical export parity, not outcome generalization or safety calibration evidence


## State of the art

XGBoost and PPO are established method families; this lesson evaluates the released Neuraxis implementation and its export path rather than claiming a new learning algorithm. Component ablations and reference-versus-optimized implementation checks are necessary evidence, but neither alone demonstrates a state-of-the-art advance.

## Results

All four ablations preserve the same 1,160/1,200 task successes, so this experiment does not show success contribution by memory, effects, deferral or invalidation. Runtime budgets differ across variants, and measured export timings exclude Python feature construction. Numerical parity is not policy safety calibration.

No generated outputs are committed by this authoring step; `scripts/run_notebooks.py` executes and stores them from a fresh kernel.

## Summary

The negative ablation result is material: the proposed components do not change success on this saturated suite. Native exports meet their recorded numerical tolerance, which supports implementation consistency only.

## References

- [XGB](https://arxiv.org/abs/1603.02754)
- [PPO](https://arxiv.org/abs/1707.06347)